In [20]:
import os
import rasterio
import json
import numpy as np

import matplotlib.pyplot as plt

# Volume sampling based on inifinite slope analysis and displacement thresholds.

This notebooks implements a simple selection of release volumes based on an empirical corelation between displacements and the ratio of yield acceleration to peak ground acceleration of the earthqueke event. Given that the displacement is larger than a given threshold, the material is assumed to fluidize.

In [ ]:
for element in slope_analysis_output:
    print(element.keys())

In [16]:
def displacement(ky, pga, M=None, pgv=None, model="scalar"):
    """ Calculation of ground displacements
    Implementation of the statistical model for ground displacements of natural slopes subject to earthquakes 
    given in [1]. Note: The statistical model is develpped for subaerial conditions.
    
    Parameters:
        ky: float or ndarray 
            Yield acceleration calculated using infinite slope analysis [g].
        pga: float or ndarray. Same dimension as ky.
            Peak ground acceleration of the event [g]. 
        M: float
            moment magnitude of the event.
        pgv: float or ndarray. Same dimension as ky.
            Peak ground velocity of the event [cm/s].
        model: string
            Different models. Options are "scalar" or "vector". If "scalar", then M must be supplied. 
            If "vector", then pgv must be supplied. Default is "scalar".
        
    Returns:
        ln(displacements) [cm], standard_deviation: float or ndarray, float or ndarray
        Logarithm of estimated displacements and associated standard deviation. 
        Standard deviation of lognormal multiplicative noise (as a function of ky/pga).  
        
    1. Rathje and Saygili, ‘Probabilistic Assessment of Earthquake-Induced Sliding Displacements of Natural Slopes’.
    """
    if model == "scalar":
        a = [-29.06, 42.49, - 19.64,-4.85, 4.89] # polynomial coefficients.
        ln_d = np.polyval(a, ky/pga) + 0.72*np.log(pga) + 0.89*(M-6)
        sigma_ln = np.polyval([-0.539, 0.789, 0.732], ky/pga)
    elif model == "vector":
        a = [-30.5, 44.75, -20.84, -4.58, -1.56]
        ln_d = np.polyval(a, ky/pga) - 0.64*np.log(pga) + 1.55*np.ln(pgv)
        sigma_ln = 0.405 + 0.524*(ky/pga)
    return(ln_d, sigma_ln)

## Plot displacements.

In [96]:
ky = np.linspace(0, 0.4, 100)
pga = np.linspace(0.01, 1., 100)
M = 6.
kys, pgas = np.meshgrid(ky, pga)

In [196]:
lnd, ln_sigma = displacement(kys, pgas, M)
d = np.exp(lnd)
log_d = lnd*np.log10(np.exp(1))
sigma = np.exp(ln_sigma)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(nrows=1, ncols=3, sharey=False)
fig.set_figwidth(14)

#d_levels = np.linspace(start=np.min(d), stop=np.max(d),num=30)
d_lnlevels = np.linspace(start=-10, stop=3,num=30)
contour1 = ax1.contourf(kys, pgas, log_d, levels=d_lnlevels, cmap=plt.cm.bone)
ax1.set_xlabel("yield acceleration [g]")
ax1.set_ylabel("pga [g]")

cbar1 = fig.colorbar(contour1, ax=ax1)
ax1.set_title('log displacements [cm]')

d_levels = np.linspace(start=np.min(d), stop=np.max(d),num=30)
contour2 = ax2.contourf(kys, pgas, d, levels=d_levels, cmap=plt.cm.bone)
ax2.set_xlabel("yield acceleration [g]")

cbar2 = fig.colorbar(contour2, ax=ax2,)
ax2.set_title('displacements [cm]')

sigma_levels = np.linspace(start=-2, stop=1.,num=30)
contour3 = ax3.contourf(kys, pgas, ln_sigma, levels=sigma_levels, cmap=plt.cm.bone)
ax3.set_xlabel("yield acceleration [g]")

cbar3 = fig.colorbar(contour3, ax=ax3)
ax3.set_title('sigma_ln')

plt.subplots_adjust(wspace=0.3)

## Computing displacement quantiles.

Suppose we are give a set of yield acceleration quantiles.

In [216]:
def write_tif(fname, data, profile):
    "Write .tif data and profile using rasterio."
    print(f"Write file: {fname}")
    with rasterio.open(fname, 'w', **profile) as dst:
        dst.write(data, 1)

In [ ]:
rundir = "/home/ebr/projects/release-volume-sampler/generated/messina_001_20240923_121008"
output_dir = os.path.join(rundir, "release_volumes")
pga=0.7
magnitude = 6.

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Load yield acceleration maps.
slope_analysis_output_folder = os.path.join(rundir, "slope_analysis")

# load json file
with open(os.path.join(slope_analysis_output_folder, "content.json"),'r') as f:
    slope_analysis_output = json.load(f)

# Read yield acceleration output from slope analysis.
yield_acceleration_quantiles = [e for e in slope_analysis_output if e["value"] == "yield_acceleration"]

output = []
for i, element in enumerate(yield_acceleration_quantiles):
    with rasterio.open(os.path.join(slope_analysis_output_folder, element["file"])) as src:
        ky = src.read(1)
        ky_profile = src.profile.copy()
        
        # Calculate displacement quantiles 
        # Here we apply the fact that displacements are decreasing with ky.
        ln_d, ln_sigma = displacement(ky=ky, pga=pga, M=magnitude)
        filename = f"d_{i}.tif"
        output.append({"file": filename, 
                       "quantile": 1. - element["quantile"], "value": "displacement", "scaling": "ln"})
        write_tif(os.path.join(output_dir, filename), ln_d, ky_profile)

content_file = os.path.join(output_dir, 'content.json')
print("Write file: {}".format(content_file))
with open(content_file, 'w') as f:
    json.dump(output, f, indent=4)

Next:
1. Create sampling proceedure for volumes by inverse transfrom sampling. Most likely, the tail is more relevant. Can I sample more of those?
2. How do I decide the depth of the volume? Can I sample that also?